# ch01 — Defining the TSAD problem

Build the anomaly types (point / contextual / collective) by hand.
Theory: [docs/learn/ch01](../docs/learn/ch01_problem.md)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tsad_forge.synthetic.generator import generate_synthetic

In [ ]:
# Synthetic data per anomaly type
fig, axes = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for ax, kind in zip(axes, ["spike", "level_shift", "contextual"]):
    ds = generate_synthetic(n_test=800, n_events=3, anomaly_kinds=[kind], seed=1)
    ax.plot(ds.test[:, 0], lw=0.8)
    ax.fill_between(np.arange(800), *ax.get_ylim(), where=ds.labels.astype(bool),
                    alpha=0.25, color="red")
    ax.set_title(f"anomaly kind = {kind}  (rate={ds.anomaly_rate:.3f})")
plt.tight_layout()

In [ ]:
# Contamination: how a polluted train split hurts the zscore baseline
from tsad_forge.models.registry import get_model
from tsad_forge.evaluation.metrics import compute_metrics

for cont in [0.0, 1.0, 3.0]:
    ds = generate_synthetic(seed=3, contamination=cont)
    scores = get_model("zscore").fit(ds.train).score(ds.test)
    m = compute_metrics(scores, ds.labels)
    print(f"contamination={cont}: VUS-PR={m['vus_pr']:.3f}  AUC-PR={m['auc_pr']:.3f}")